In [0]:
# Create a text widget named "catalog" with default value "new_catalog"
dbutils.widgets.text("catalog", "new_catalog")

# Retrieve the value entered in the "catalog" widget, strip whitespace, and store in variable Catalog1
Catalog1 = dbutils.widgets.get("catalog").strip()

# Create a text widget named "schema" with default value "default_schema"
dbutils.widgets.text("schema", "default_schema")

# Retrieve the value entered in the "schema" widget, strip whitespace, and store in variable Schema1
Schema1 = dbutils.widgets.get("schema").strip()

In [0]:
import json

# Run the common configuration notebook with a 360-second timeout
# Pass in dynamic parameters for catalog and schema (from widgets)
json_obj = dbutils.notebook.run(
    "/Workspace/Users/viggneshwar@gmail.com/databricks/Logistics/Project/Generic_Function/common_config_nb",
    360,
    {"catalog_new": Catalog1, "schema_new": Schema1}
)

# Parse the JSON string returned by the notebook into a Python dictionary
config_dict = json.loads(json_obj)

# Extract key configuration values from the dictionary
bronze_path = config_dict["bronze_path"]        # Path for bronze layer data
source_path = config_dict["sourcedata_path"]    # Path for source/raw data
gold_path   = config_dict["gold_path"]          # Path for gold layer data
gold_db     = config_dict["gold_db"]            # Database/schema for gold layer tables
silver_db   = config_dict["silver_db"]          # Database/schema for silver layer tables

In [0]:
spark.sql(f"""CREATE Table if not EXISTS 
{gold_db}.driver_app_data
as 
select full_name, role, origin_hub_city from {gold_db}.staff_gold_tbl""")
     

In [0]:
spark.sql(f"""CREATE TABLE IF NOT EXISTS {gold_db}.active_operational_problem_data
as select * from {gold_db}.logistics_shipment_gold_curated_tbl where shipment_status IN ('DELAYED','RETURNED')""")

In [0]:
spark.sql(f"""CREATE Table if not EXISTS 
{gold_db}.senior_insurance_audit
as 
select * from {gold_db}.staff_gold_tbl where age >50""")

In [0]:
spark.sql(f"""CREATE TABLE IF NOT EXISTS {gold_db}.regional_staffing_analysis as 
select origin_hub_city, count(*) as staff_count from {gold_db}.staff_gold_tbl group by origin_hub_city""")

In [0]:
spark.sql(f"""CREATE TABLE IF NOT EXISTS {gold_db}.fleet_capacity_analysis as 
select shipment_vehicle_type, sum(shipment_weight_kg) as total_weight from {gold_db}.logistics_shipment_gold_curated_tbl group by shipment_vehicle_type""")
     